# Import Library

In [18]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
print("LOAD DATA BERSIH")
df = pd.read_csv('../data/processed/clean_data_games.csv')

df['tags_clean'] = df['tags_clean'].fillna('')
display(df.head())

LOAD DATA BERSIH


,app_id,title,description,tags,rating,desc_sentence,tags_clean
0,13500,Prince of Persia: Warrior Within™,Enter the dark underworld of Prince of Persia ...,"['Action', 'Adventure', 'Parkour', 'Third Pers...",Very Positive,enter dark underworld prince persia warrior wi...,Action Adventure Parkour Third Person Great So...
1,113020,Monaco: What's Yours Is Mine,Monaco: What's Yours Is Mine is a single playe...,"['Co-op', 'Stealth', 'Indie', 'Heist', 'Local ...",Very Positive,monaco whats mine single player coop heist gam...,Coop Stealth Indie Heist Local CoOp Strategy O...
2,226560,Escape Dead Island,Escape Dead Island is a Survival-Mystery adven...,"['Zombies', 'Adventure', 'Survival', 'Action',...",Mixed,escape dead island survivalmystery adventure l...,Zombies Adventure Survival Action Third Person...
3,249050,Dungeon of the ENDLESS™,Dungeon of the Endless is a Rogue-Like Dungeon...,"['Roguelike', 'Strategy', 'Tower Defense', 'Pi...",Very Positive,dungeon endless roguelike dungeondefense game ...,Roguelike Strategy Tower Defense Pixel Graphic...
4,250180,METAL SLUG 3,"“METAL SLUG 3”, the masterpiece in SNK’s emble...","['Arcade', 'Classic', 'Action', 'Co-op', 'Side...",Very Positive,metal slug masterpiece snks emblematic run gun...,Arcade Classic Action Coop Side Scroller Retro...


# Proses TF-IDF (Term Frequency-Inverse Document Frequency)

In [20]:

print("MEMBUAT MATRIKS TF-IDF")
tfidf = TfidfVectorizer()

# Mengubah kolom tags_clean menjadi matriks angka
tfidf_matrix = tfidf.fit_transform(df['tags_clean'])

print(f"Ukuran Matriks TF-IDF: {tfidf_matrix.shape}")

MEMBUAT MATRIKS TF-IDF
Ukuran Matriks TF-IDF: (40499, 471)


## Fungsi rekomendasi

In [23]:
def get_recommendation(game_title, df, tfidf_matrix, top_n=5):
    # 1. Cari index game yang di-input user
    try:
        idx = df[df['title'].str.lower() == game_title.lower()].index[0]
    except IndexError:
        return f"Game '{game_title}' tidak ditemukan di database."

    # 2. Ambil vektor TF-IDF khusus dari game tersebut
    game_vector = tfidf_matrix[idx]

    # 3. Hitung Cosine Similarity game ini dengan game lain
    sim_scores = cosine_similarity(game_vector, tfidf_matrix).flatten()

    # 4. Urutkan dari yang paling mirip
    similar_indices = sim_scores.argsort()[::-1]

    # 5. Top N game
    top_indices = similar_indices[1:top_n+1]

    # 6. Tampilkan hasil
    recom_df = df.iloc[top_indices][['title', 'tags_clean', 'desc_sentence', 'rating']].copy()
    recom_df['similarity_score'] = sim_scores[top_indices] 
    
    return recom_df

# Test rekomendasi

In [24]:
print("Mencari rekomendasi untuk 'METAL SLUG 3'...")
display(get_recommendation("METAL SLUG 3", df, tfidf_matrix, top_n=5))

Mencari rekomendasi untuk 'METAL SLUG 3'...


,title,tags_clean,desc_sentence,rating,similarity_score
11831,METAL SLUG X,Action Arcade Coop Classic Shoot Em Up D Retro...,metal slug x one highly praised title series a...,Very Positive,0.852611
13287,METAL SLUG 2,Action Arcade D Side Scroller Retro Classic Sh...,metal slug nd entry snks emblematic run gun ac...,Very Positive,0.765203
3705,METAL SLUG,Action Arcade D Shoot Em Up Retro Classic Side...,metal slug first title snks legendary run gun ...,Very Positive,0.746911
1973,Contra Anniversary Collection,Action Side Scroller D Retro Classic Platforme...,contra anniversary collection brings classic r...,Mostly Positive,0.734256
5522,Super Cyborg,Action Indie Retro D Shooter Platformer Pixel ...,super cyborg old school nonstop hardcore runng...,Very Positive,0.718918


In [27]:
# Cek banyak game yang tag-nya kosong
jumlah_kosong = len(df[df['tags_clean'] == ''])
print(f"Total game dengan tag kosong: {jumlah_kosong} dari {len(df)} game")

Total game dengan tag kosong: 15 dari 40499 game


In [29]:
# buang game dengan tag kosong
df = df[df['tags_clean'] != '']

df = df.reset_index(drop=True)

tfidf_matrix = tfidf.fit_transform(df['tags_clean'])

print(f"Pembersihan selesai. Sisa data sekarang: {len(df)} game.")
print(f"Ukuran Matriks TF-IDF baru: {tfidf_matrix.shape}")

Pembersihan selesai. Sisa data sekarang: 40484 game.
Ukuran Matriks TF-IDF baru: (40484, 471)


# export model

In [31]:
import pickle
import os

os.makedirs('../models', exist_ok=True)

with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('../models/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

df.to_pickle('../models/clean_games_df.pkl')

print("Berhasil menyimpan model")

Berhasil menyimpan model
